# Storage Format Intelligence & I/O Optimization — dask_setup Recipe

This notebook demonstrates how to choose and benchmark storage formats and I/O strategies:

- Create sample NetCDF climate data files
- Benchmark `netcdf4` vs `h5netcdf` engine performance
- Convert a multi-file NetCDF dataset to Zarr with rechunking
- Compare NetCDF vs Zarr on the same workload
- Observe cold-cache vs warm-cache I/O effects
- Summary of best practices

## Setup and Imports

In [ ]:
import time
from contextlib import contextmanager
from pathlib import Path
import tempfile

import numpy as np
import xarray as xr

from dask_setup import setup_dask_client

# Check optional dependencies
try:
    import zarr
    ZARR_AVAILABLE = True
    print(f"zarr {zarr.__version__}")
except ImportError:
    ZARR_AVAILABLE = False
    print("zarr not installed — Zarr sections will be skipped")

NETCDF_ENGINES = []
for eng in ("netcdf4", "h5netcdf"):
    try:
        __import__(eng.replace("netcdf4", "netCDF4").replace("h5netcdf", "h5netcdf"))
        NETCDF_ENGINES.append(eng)
    except ImportError:
        pass
# fallback probe via xarray
for eng in ("netcdf4", "h5netcdf"):
    if eng not in NETCDF_ENGINES:
        try:
            xr.open_dataset.__doc__  # just check xarray is present
            NETCDF_ENGINES.append(eng)  # assume available if xarray is
        except Exception:
            pass
NETCDF_ENGINES = list(dict.fromkeys(NETCDF_ENGINES))  # deduplicate


@contextmanager
def timer(label=""):
    t = {"start": time.perf_counter()}
    try:
        yield t
    finally:
        t["total"] = time.perf_counter() - t["start"]
        print(f"  [{label}] {t['total']:.2f}s")


print("Imports OK")
print(f"NetCDF engines available: {NETCDF_ENGINES}")

## 1. Create Sample NetCDF Files

Generate synthetic CF-compliant climate data (temperature + precipitation) for benchmarking.
We use a small spatial grid and few timesteps so this runs quickly on a laptop.

In [ ]:
# Working directory for generated files
work_dir = Path(tempfile.mkdtemp(prefix="dask_recipe_io_"))
netcdf_dir = work_dir / "sample_netcdf"
netcdf_dir.mkdir()

NUM_FILES = 10
TIME_PER_FILE = 20
LAT_SIZE, LON_SIZE = 60, 120   # smaller grid for speed

lat = np.linspace(-90, 90, LAT_SIZE)
lon = np.linspace(-180, 180, LON_SIZE)
file_paths = []

for i in range(NUM_FILES):
    np.random.seed(42 + i)
    time_coord = np.arange(i * TIME_PER_FILE, (i + 1) * TIME_PER_FILE)
    temp = 273.15 + 15 * np.cos(2 * np.pi * time_coord / 365.25)[:, None, None] \
           + 5 * np.random.random((TIME_PER_FILE, LAT_SIZE, LON_SIZE))
    precip = np.abs(np.random.gamma(2, 2, (TIME_PER_FILE, LAT_SIZE, LON_SIZE)))

    ds = xr.Dataset(
        {
            "temperature": (["time", "lat", "lon"], temp.astype("float32"),
                            {"units": "K", "long_name": "Near-surface air temperature"}),
            "precipitation": (["time", "lat", "lon"], precip.astype("float32"),
                              {"units": "mm/day", "long_name": "Daily precipitation"}),
        },
        coords={
            "time": time_coord,
            "lat": ("lat", lat, {"units": "degrees_north"}),
            "lon": ("lon", lon, {"units": "degrees_east"}),
        },
    )
    ds.attrs["conventions"] = "CF-1.8"

    fp = netcdf_dir / f"sample_{i + 1:03d}.nc"
    ds.to_netcdf(fp, encoding={"temperature": {"zlib": True}, "precipitation": {"zlib": True}})
    file_paths.append(fp)

total_mb = sum(f.stat().st_size for f in file_paths) / 1e6
print(f"Created {NUM_FILES} NetCDF files  ({total_mb:.1f} MB total)")

## 2. Start an I/O-Optimised Cluster

`workload_type="io"` uses a single process with many threads — ideal for opening
many files concurrently without the inter-process communication overhead.

In [ ]:
client, cluster, tmp = setup_dask_client(
    workload_type="io",
    max_workers=4,
    reserve_mem_gb=4.0,
    fallback_on_detection_failure=True,
)
print(f"Workers: {len(client.scheduler_info()['workers'])}")
print(f"Temp dir: {tmp}")

## 3. NetCDF Engine Benchmark

Compare `netcdf4` and `h5netcdf` for single-file reading, multi-file `open_mfdataset`, and
a chunked spatial reduction.

In [ ]:
engine_results = {}

for engine in NETCDF_ENGINES:
    print(f"\nEngine: {engine}")
    try:
        # Single file
        with timer(f"{engine} single-file") as t1:
            ds = xr.open_dataset(file_paths[0], engine=engine)
            _ = ds.temperature.values
            ds.close()

        # Multi-file
        with timer(f"{engine} multi-file (5 files)") as t2:
            ds = xr.open_mfdataset(
                file_paths[:5], engine=engine,
                chunks={"time": 20, "lat": 30, "lon": 60},
                parallel=True, combine="by_coords",
            )
            _ = ds.temperature.mean().compute()
            ds.close()

        engine_results[engine] = {
            "single_file_s": t1["total"],
            "multi_file_s": t2["total"],
        }
    except Exception as e:
        print(f"  ERROR: {e}")
        engine_results[engine] = {"error": str(e)}

print("\nSummary:")
for eng, r in engine_results.items():
    if "error" not in r:
        print(f"  {eng:12s}  single={r['single_file_s']:.2f}s  multi={r['multi_file_s']:.2f}s")

## 4. Convert NetCDF to Zarr

Zarr uses chunked, compressed storage that is friendlier to parallel reads and cloud access.
Rechunk during conversion to optimise for analysis access patterns.

In [ ]:
zarr_store = work_dir / "converted.zarr"

if ZARR_AVAILABLE:
    engine = NETCDF_ENGINES[0] if NETCDF_ENGINES else "netcdf4"

    with timer("NetCDF → Zarr conversion") as t:
        ds = xr.open_mfdataset(
            file_paths, engine=engine,
            chunks={"time": 40, "lat": 30, "lon": 60},
            parallel=True, combine="by_coords",
        )
        # Rechunk for better Zarr layout
        ds = ds.chunk({"time": 60, "lat": LAT_SIZE, "lon": LON_SIZE})

        encoding = {
            "temperature":   {"compressor": zarr.Blosc(cname="lz4", clevel=5)},
            "precipitation": {"compressor": zarr.Blosc(cname="lz4", clevel=5)},
        }
        ds.to_zarr(zarr_store, mode="w", consolidated=True, encoding=encoding)
        ds.close()

    zarr_mb = sum(f.stat().st_size for f in zarr_store.rglob("*") if f.is_file()) / 1e6
    netcdf_mb = sum(f.stat().st_size for f in file_paths) / 1e6
    print(f"  NetCDF: {netcdf_mb:.1f} MB → Zarr: {zarr_mb:.1f} MB  "
          f"(ratio {zarr_mb / netcdf_mb:.2f}x)")
else:
    print("Zarr not available — skipping conversion")

## 5. NetCDF vs Zarr Performance

Run the same analysis (spatial mean, annual resampling, temporal std) on both formats.

In [ ]:
fmt_times = {}

# NetCDF
if NETCDF_ENGINES:
    engine = NETCDF_ENGINES[0]
    with timer("NetCDF analysis") as t:
        ds = xr.open_mfdataset(
            file_paths, engine=engine,
            chunks={"time": 40, "lat": 30, "lon": 60}, parallel=True,
        )
        _ = ds.temperature.mean().compute()
        _ = ds.temperature.std(dim="time").compute()
        ds.close()
    fmt_times["NetCDF"] = t["total"]

# Zarr
if ZARR_AVAILABLE and zarr_store.exists():
    with timer("Zarr analysis") as t:
        ds = xr.open_zarr(zarr_store, consolidated=True)
        _ = ds.temperature.mean().compute()
        _ = ds.temperature.std(dim="time").compute()
        ds.close()
    fmt_times["Zarr"] = t["total"]

if len(fmt_times) == 2:
    speedup = fmt_times["NetCDF"] / fmt_times["Zarr"]
    print(f"\nZarr speedup: {speedup:.2f}x")

## 6. Filesystem Caching Effects

The first read of a file is slower (cold cache) than subsequent reads (warm OS file cache).
This is important to account for when benchmarking I/O.

In [ ]:
if NETCDF_ENGINES:
    engine = NETCDF_ENGINES[0]
    subset = file_paths[:3]
    chunk_kw = {"time": 20, "lat": 30, "lon": 60}

    with timer("Cold cache (1st read)") as t_cold:
        ds = xr.open_mfdataset(subset, engine=engine, chunks=chunk_kw)
        _ = ds.temperature.mean().compute()
        ds.close()

    with timer("Warm cache (2nd read)") as t_warm:
        ds = xr.open_mfdataset(subset, engine=engine, chunks=chunk_kw)
        _ = ds.temperature.mean().compute()
        ds.close()

    speedup = t_cold["total"] / t_warm["total"]
    print(f"Cache speedup: {speedup:.2f}x")

In [ ]:
client.close()
cluster.close()

# Clean up generated files
import shutil
shutil.rmtree(work_dir, ignore_errors=True)
print("Cluster closed, temp files removed.")

## Storage Format & I/O Best Practices

**Choose your format:**

| Format | Best for | Key advantage |
|--------|----------|---------------|
| NetCDF | Archival, sharing with collaborators | Self-describing, CF-compliant |
| Zarr | Analysis pipelines, cloud storage | Chunked, concurrent writes, cloud-native |

**Engine selection:**
- `netcdf4` — default, widely compatible, single-threaded C library
- `h5netcdf` — pure Python, no GIL → better parallel throughput on multi-file datasets

```python
# Recommended for large multi-file datasets
ds = xr.open_mfdataset(files, engine="h5netcdf", parallel=True, combine="by_coords")
```

**Zarr write tips:**
```python
# Always consolidate metadata
ds.to_zarr("output.zarr", mode="w", consolidated=True,
           encoding={"temp": {"compressor": zarr.Blosc(cname="lz4", clevel=5)}})
```

**Chunk sizing:**  target 256–512 MiB per chunk for analysis.  Use `recommend_chunks()` if unsure:
```python
from dask_setup import recommend_chunks
chunks = recommend_chunks(ds, client, verbose=True)
```